# boxscore_backfill Notebook

**Issue #172** — Interactive backfill of `fact_boxscore_game_stats`, `fact_skater_stats`,
and `fact_goalie_stats` for a configurable range of historical games.

Follows the `dim_player_backfill.ipynb` pattern (Issue #166):
- Imports SQLAlchemy models directly from `../backend/models.py` — no Flask app context
- Connects to `instance/nhl.db` via `create_engine`
- Commits once per game to limit data loss on interruption
- 50 ms rate-limiting between API requests

## Run instructions

```bash
pip install jupyter pandas httpx sqlalchemy pytz
jupyter notebook nhl-dashboard/notebooks/boxscore_backfill.ipynb
```

Run all cells top-to-bottom. Edit only the **Config** cell before running.

## Notebook structure

| Section | Content |
|---|---|
| Config | `START_DATE`, `END_DATE`, `GAME_TYPES` — only cell to edit |
| Setup | Imports, SQLAlchemy engine, session factory |
| Section 1 | Load game IDs from `game` table for the configured date range |
| Section 2 | Preview DataFrame of games to process; confirm before writing |
| Section 3 | `upsert_game_stats()`, `upsert_skater_stats()`, `upsert_goalie_stats()`, `check_and_backfill_player()` |
| Section 4 | Batch upsert loop over game IDs with progress output |
| Section 5 | Verification row counts and sample rows for all three fact tables |

## Config

**Edit this cell only.** Set the date range and game types to backfill.
`GAME_TYPES`: `2` = regular season, `3` = playoffs.

In [ ]:
# START_DATE  = "2025-10-01"   # YYYY-MM-DD — first game date to include
# END_DATE    = "2026-04-18"   # YYYY-MM-DD — last game date to include
# GAME_TYPES  = [2, 3]         # 2 = regular season, 3 = playoffs

# print(f"Date range : {START_DATE} → {END_DATE}")
# print(f"Game types : {GAME_TYPES}")

## Setup

Imports, SQLAlchemy connection to `instance/nhl.db`, and the NHL API base URL.
No Flask app context — models are imported directly from `../backend/models.py`.

In [ ]:
import sys
import time
from datetime import datetime
from pathlib import Path

import httpx
import pandas as pd
import pytz
from sqlalchemy import create_engine, text
from sqlalchemy.orm import sessionmaker

# Add backend to path so we can import models without Flask app context
BACKEND_DIR = Path("../backend").resolve()
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

NHL_BASE = "https://api-web.nhle.com/v1"
DB_PATH  = BACKEND_DIR / "instance" / "nhl.db"

# Connect directly to the Flask app's SQLite DB — no app context needed
engine  = create_engine(f"sqlite:///{DB_PATH}", echo=False, connect_args={"timeout": 30})
Session = sessionmaker(bind=engine, autoflush=False)

ET = pytz.timezone("US/Eastern")


def now_eastern() -> datetime:
    """Return the current datetime in US/Eastern (naive, tzinfo stripped)."""
    return datetime.now(ET).replace(tzinfo=None)


print(f"Database  : {DB_PATH}")
print(f"DB exists : {DB_PATH.exists()}")

In [83]:
with engine.connect() as conn:
    rows = conn.execute(text("SELECT DISTINCT season FROM game ORDER BY season")).fetchall()

SEASONS = [row[0] for row in rows]
# SEASONS = SEASONS[-1:]
print(f"Seasons ({len(SEASONS)}): {SEASONS}")

Seasons (109): [19171918, 19181919, 19191920, 19201921, 19211922, 19221923, 19231924, 19241925, 19251926, 19261927, 19271928, 19281929, 19291930, 19301931, 19311932, 19321933, 19331934, 19341935, 19351936, 19361937, 19371938, 19381939, 19391940, 19401941, 19411942, 19421943, 19431944, 19441945, 19451946, 19461947, 19471948, 19481949, 19491950, 19501951, 19511952, 19521953, 19531954, 19541955, 19551956, 19561957, 19571958, 19581959, 19591960, 19601961, 19611962, 19621963, 19631964, 19641965, 19651966, 19661967, 19671968, 19681969, 19691970, 19701971, 19711972, 19721973, 19731974, 19741975, 19751976, 19761977, 19771978, 19781979, 19791980, 19801981, 19811982, 19821983, 19831984, 19841985, 19851986, 19861987, 19871988, 19881989, 19891990, 19901991, 19911992, 19921993, 19931994, 19941995, 19951996, 19961997, 19971998, 19981999, 19992000, 20002001, 20012002, 20022003, 20032004, 20042005, 20052006, 20062007, 20072008, 20082009, 20092010, 20102011, 20112012, 20122013, 20132014, 20142015, 2015

## Section 1 — Load game IDs from DB

Queries the `game` table for all game IDs where `game_date` falls within the
configured date range and `game_type` is in `GAME_TYPES`.

Joins to the `team` table on `team_id` to resolve away/home tri-codes for the
Section 2 preview.

In [88]:
# Ensure SEASONS is a list of ints (safe for assembling the IN-list)
SEASONS = [int(s) for s in SEASONS]
SEASONS_PLACEHOLDER = ",".join(str(season) for season in SEASONS)
_sql = f"""
    SELECT
        g.game_id,
        g.game_date,
        COALESCE(at.tri_code, CAST(g.away_team_id AS TEXT)) AS away_team,
        COALESCE(ht.tri_code, CAST(g.home_team_id AS TEXT)) AS home_team,
        g.game_state_id
    FROM game g
    LEFT JOIN team at ON at.team_id = g.away_team_id
    LEFT JOIN team ht ON ht.team_id = g.home_team_id
    LEFT JOIN boxscore b ON b.game_id = g.game_id
    WHERE season IN ({SEASONS_PLACEHOLDER})
        AND g.game_type NOT IN (1)  -- regular season and playoffs only
        AND b.game_id IS NULL  -- only include games missing boxscore data
    ORDER BY g.game_date, g.game_id
"""

with engine.connect() as conn:
    _rows = conn.execute(
        text(_sql),
        {"SEASONS": SEASONS},
    ).fetchall()

GAME_IDS    = [row[0] for row in _rows]
_GAME_ROWS  = _rows   # kept for Section 2 preview

print(f"Games found in DB for {SEASONS_PLACEHOLDER} : {len(GAME_IDS)}")

Games found in DB for 19171918,19181919,19191920,19201921,19211922,19221923,19231924,19241925,19251926,19261927,19271928,19281929,19291930,19301931,19311932,19321933,19331934,19341935,19351936,19361937,19371938,19381939,19391940,19401941,19411942,19421943,19431944,19441945,19451946,19461947,19471948,19481949,19491950,19501951,19511952,19521953,19531954,19541955,19551956,19561957,19571958,19581959,19591960,19601961,19611962,19621963,19631964,19641965,19651966,19661967,19671968,19681969,19691970,19701971,19711972,19721973,19731974,19741975,19751976,19761977,19771978,19781979,19791980,19801981,19811982,19821983,19831984,19841985,19851986,19861987,19871988,19881989,19891990,19901991,19911992,19921993,19931994,19941995,19951996,19961997,19971998,19981999,19992000,20002001,20012002,20022003,20032004,20042005,20052006,20062007,20072008,20082009,20092010,20102011,20112012,20122013,20132014,20142015,20152016,20162017,20172018,20182019,20192020,20202021,20212022,20222023,20232024,20242025,202520

In [ ]:
# Ensure SEASONS is a list of ints (safe for assembling the IN-list)
SEASONS = [int(s) for s in SEASONS]
SEASONS_PLACEHOLDER = ",".join(str(season) for season in SEASONS)
_sql = f"""
    SELECT
        g.game_id,
        g.game_date,
        COALESCE(at.tri_code, CAST(g.away_team_id AS TEXT)) AS away_team,
        COALESCE(ht.tri_code, CAST(g.home_team_id AS TEXT)) AS home_team,
        g.game_state_id
    FROM game g
    LEFT JOIN team at ON at.team_id = g.away_team_id
    LEFT JOIN team ht ON ht.team_id = g.home_team_id
    WHERE season IN ({SEASONS_PLACEHOLDER})
        AND g.game_type NOT IN (1)  -- regular season and playoffs only
    ORDER BY g.game_date, g.game_id
"""

with engine.connect() as conn:
    _rows = conn.execute(
        text(_sql),
        {"SEASONS": SEASONS},
    ).fetchall()

GAME_IDS    = [row[0] for row in _rows]
_GAME_ROWS  = _rows   # kept for Section 2 preview

print(f"Games found in DB for {SEASONS_PLACEHOLDER} : {len(GAME_IDS)}")

## Section 2 — Preview

Displays a DataFrame of all games to be processed — with game date, away team,
home team, and game state ID — so the developer can confirm the scope before
any upserts run.

**Review this output before proceeding to Section 4.**

In [85]:
df_games = pd.DataFrame(
    _GAME_ROWS,
    columns=["game_id", "game_date", "away_team", "home_team", "game_state_id"],
)

print(f"Games to be processed: {len(df_games)}")
display(df_games.head(20))

Games to be processed: 61553


,game_id,game_date,away_team,home_team,game_state_id
0,1917020001,1917-12-19,MTL,SEN,7
1,1917020002,1917-12-19,TAN,MWN,7
2,1917020003,1917-12-22,SEN,TAN,7
3,1917020004,1917-12-22,MTL,MWN,7
4,1917020005,1917-12-26,SEN,MWN,7
5,1917020006,1917-12-26,MTL,TAN,7
6,1917020007,1917-12-29,MWN,SEN,7
7,1917020008,1917-12-29,TAN,MTL,7
8,1917020009,1918-01-02,TAN,SEN,7
9,1917020035,1918-01-02,MWN,MTL,7


## Section 3 — Upsert functions

Defines four functions used by the Section 4 batch loop:

- `check_and_backfill_player(session, player_id)` — ensures `dim_player` has a row
  for `player_id`; fetches biographical data from the player landing endpoint if not
- `upsert_game_stats(session, game_id, raw)` — wraps `FactBoxscoreGameStats` upsert
  (Issue #170 persist logic)
- `upsert_skater_stats(session, game_id, raw)` — wraps `FactSkaterStats` upsert
  (Issue #169 persist logic)
- `upsert_goalie_stats(session, game_id, raw)` — wraps `FactGoalieStats` upsert
  (Issue #169 persist logic); parses `saveShotsAgainst` into separate `saves` and
  `shots_against` integers

Models are imported here to keep column definitions in sync with the live schema.

In [86]:
from models import DimPlayer, FactBoxscoreGameStats, FactSkaterStats, FactGoalieStats

_PERIOD_ORDINALS = {1: "1st", 2: "2nd", 3: "3rd"}


def _parse_period(period_descriptor: dict) -> str | None:
    """Convert a periodDescriptor dict to a human-readable period string."""
    if not period_descriptor:
        return None
    period_type = period_descriptor.get("periodType", "REG")
    period_num  = period_descriptor.get("number", 1)
    if period_type == "OT":
        return "OT"
    if period_type == "SO":
        return "SO"
    return _PERIOD_ORDINALS.get(period_num, f"{period_num}th")


def _parse_save_shots(save_shots_against: str | None) -> tuple:
    """Parse '29/30' saveShotsAgainst into (saves=29, shots_against=30)."""
    if not save_shots_against:
        return None, None
    try:
        parts = save_shots_against.split("/")
        return int(parts[0]), int(parts[1])
    except (ValueError, IndexError):
        return None, None


def check_and_backfill_player(session, player_id: int) -> bool:
    """Return True if dim_player has a row for player_id; backfill from the
    player landing endpoint if not.

    Consistent with the detection logic in Issue #169. If the backfill API
    call fails, logs the error and returns False so the caller can skip the
    fact row and avoid an orphaned FK.

    Args:
        session: SQLAlchemy Session bound to instance/nhl.db.
        player_id: NHL numeric player identifier.

    Returns:
        True when a dim_player row exists (or was just created), False on failure.
    """
    if session.get(DimPlayer, player_id):
        return True
    try:
        r = httpx.get(f"{NHL_BASE}/player/{player_id}/landing", timeout=15)
        r.raise_for_status()
        raw = r.json()
        first = raw.get("firstName", {})
        last  = raw.get("lastName", {})
        first_name = first.get("default", "") if isinstance(first, dict) else (first or "")
        last_name  = last.get("default", "")  if isinstance(last, dict)  else (last or "")
        player = DimPlayer(
            player_id        = player_id,
            first_name       = first_name,
            last_name        = last_name,
            sweater_number   = raw.get("sweaterNumber"),
            position         = raw.get("position"),
            shoots_catches   = raw.get("shootsCatches"),
            height_in_inches = raw.get("heightInInches"),
            weight_in_pounds = raw.get("weightInPounds"),
            birth_date       = raw.get("birthDate"),
            birth_country    = raw.get("birthCountry"),
            headshot_url     = raw.get("headshot"),
            updated_at       = now_eastern(),
        )
        session.merge(player)
        print(f"    [backfill] dim_player player_id={player_id}: {first_name} {last_name}")
        return True
    except Exception as exc:
        print(f"    [backfill] FAILED player_id={player_id}: {exc}")
        return False


def upsert_game_stats(session, game_id: int, raw: dict) -> None:
    """Upsert one FactBoxscoreGameStats row from a raw boxscore API response.

    Wraps the persist logic from Issue #170. Uses session.merge() so repeated
    runs on the same game_id update the row in place without duplicating it.

    Args:
        session: SQLAlchemy Session bound to instance/nhl.db.
        game_id: NHL game identifier (must match raw['id']).
        raw: Full /v1/gamecenter/{id}/boxscore API response dict.
    """
    if not raw or "id" not in raw:
        return

    venue_raw = raw.get("venue", "")
    venue = venue_raw.get("default", "") if isinstance(venue_raw, dict) else (venue_raw or "")

    start_raw = raw.get("startTimeUTC", "")
    try:
        start_utc = datetime.fromisoformat(start_raw.replace("Z", "+00:00"))
        start_est = start_utc.astimezone(ET).replace(tzinfo=None)
    except Exception:
        start_est = None

    away = raw.get("awayTeam", {})
    home = raw.get("homeTeam", {})
    away_name_raw = away.get("name", {})
    home_name_raw = home.get("name", {})
    away_name = away_name_raw.get("default", "") if isinstance(away_name_raw, dict) else (away_name_raw or "")
    home_name = home_name_raw.get("default", "") if isinstance(home_name_raw, dict) else (home_name_raw or "")

    period   = _parse_period(raw.get("periodDescriptor") or {})
    clock_raw = raw.get("clock") or {}
    clock    = clock_raw.get("timeRemaining")

    row = FactBoxscoreGameStats(
        game_id        = raw["id"],
        season_id      = raw.get("season"),
        game_type      = raw.get("gameType"),
        game_date      = raw.get("gameDate"),
        venue          = venue,
        start_time_est = start_est,
        game_state     = raw.get("gameState"),
        away_team_id   = away.get("id"),
        away_abbrev    = away.get("abbrev"),
        away_name      = away_name,
        away_score     = away.get("score"),
        away_sog       = away.get("sog"),
        home_team_id   = home.get("id"),
        home_abbrev    = home.get("abbrev"),
        home_name      = home_name,
        home_score     = home.get("score"),
        home_sog       = home.get("sog"),
        clock          = clock,
        period         = period,
    )
    session.merge(row)


def upsert_skater_stats(session, game_id: int, raw: dict) -> int:
    """Upsert FactSkaterStats rows for all forwards and defensemen in a boxscore.

    Wraps the persist logic from Issue #169. Calls check_and_backfill_player()
    before writing each fact row — consistent with the dim_player gap detection
    pattern specified in Issue #169. Players whose backfill fails are skipped.

    Args:
        session: SQLAlchemy Session bound to instance/nhl.db.
        game_id: NHL game identifier.
        raw: Full /v1/gamecenter/{id}/boxscore API response dict.

    Returns:
        Number of FactSkaterStats rows successfully upserted.
    """
    pbgs         = raw.get("playerByGameStats", {})
    away_team_id = raw.get("awayTeam", {}).get("id")
    home_team_id = raw.get("homeTeam", {}).get("id")

    sides = [
        ("away", away_team_id, pbgs.get("awayTeam", {})),
        ("home", home_team_id, pbgs.get("homeTeam", {})),
    ]

    count = 0
    for side, team_id, side_data in sides:
        for group in ("forwards", "defense"):
            for p in side_data.get(group, []):
                player_id = p.get("playerId")
                if player_id is None:
                    continue
                if not check_and_backfill_player(session, player_id):
                    continue
                row = FactSkaterStats(
                    game_id         = game_id,
                    player_id       = player_id,
                    team_id         = team_id,
                    side            = side,
                    position_group  = group,
                    position        = p.get("position"),
                    goals           = p.get("goals"),
                    assists         = p.get("assists"),
                    points          = p.get("points"),
                    plus_minus      = p.get("plusMinus"),
                    pim             = p.get("pim"),
                    toi             = p.get("toi"),
                    hits            = p.get("hits"),
                    blocked_shots   = p.get("blockedShots"),
                    pp_goals        = p.get("powerPlayGoals"),
                    pp_points       = p.get("powerPlayPoints"),
                    sh_goals        = p.get("shorthandedGoals"),
                    faceoff_win_pct = p.get("faceoffWinningPctg"),
                    giveaways       = p.get("giveaways"),
                    takeaways       = p.get("takeaways"),
                    shifts          = p.get("shifts"),
                )
                session.merge(row)
                count += 1
    return count


def upsert_goalie_stats(session, game_id: int, raw: dict) -> int:
    """Upsert FactGoalieStats rows for all goalies in a boxscore.

    Wraps the persist logic from Issue #169. Parses the `saveShotsAgainst`
    composite string ('saves/shots') into separate `saves` and `shots_against`
    integer columns. `decision` is stored as TEXT (W/L/OTL) or NULL for backup
    goalies who received no decision.

    Args:
        session: SQLAlchemy Session bound to instance/nhl.db.
        game_id: NHL game identifier.
        raw: Full /v1/gamecenter/{id}/boxscore API response dict.

    Returns:
        Number of FactGoalieStats rows successfully upserted.
    """
    pbgs         = raw.get("playerByGameStats", {})
    away_team_id = raw.get("awayTeam", {}).get("id")
    home_team_id = raw.get("homeTeam", {}).get("id")

    sides = [
        ("away", away_team_id, pbgs.get("awayTeam", {})),
        ("home", home_team_id, pbgs.get("homeTeam", {})),
    ]

    count = 0
    for side, team_id, side_data in sides:
        for p in side_data.get("goalies", []):
            player_id = p.get("playerId")
            if player_id is None:
                continue
            if not check_and_backfill_player(session, player_id):
                continue
            saves, shots_against = _parse_save_shots(p.get("saveShotsAgainst"))
            row = FactGoalieStats(
                game_id          = game_id,
                player_id        = player_id,
                team_id          = team_id,
                side             = side,
                starter          = p.get("starter"),
                toi              = p.get("toi"),
                goals_against    = p.get("goalsAgainst"),
                saves            = saves,
                shots_against    = shots_against,
                save_pct         = p.get("savePctg"),
                es_shots_against = p.get("evenStrengthShotsAgainst"),
                pp_shots_against = p.get("powerPlayShotsAgainst"),
                sh_shots_against = p.get("shorthandedShotsAgainst"),
                pim              = p.get("pim"),
                decision         = p.get("decision"),
            )
            session.merge(row)
            count += 1
    return count


print("Upsert functions defined:")
for fn in ["check_and_backfill_player", "upsert_game_stats",
           "upsert_skater_stats", "upsert_goalie_stats"]:
    print(f"  {fn}()")

Upsert functions defined:
  check_and_backfill_player()
  upsert_game_stats()
  upsert_skater_stats()
  upsert_goalie_stats()


## Section 4 — Batch upsert

Loops over all `GAME_IDS` from Section 1. For each game:

1. Fetches `/v1/gamecenter/{game_id}/boxscore` from the NHL API
2. Calls `upsert_game_stats()`, `upsert_skater_stats()`, and `upsert_goalie_stats()`
3. Commits once per game — an interrupted run can be safely re-run; already-ingested
   rows are updated in place via `UNIQUE(game_id, player_id)` upsert semantics
4. Rate-limits at 50 ms between requests

Individual API failures are logged and skipped without aborting the loop.

In [87]:
processed = 0
failed    = 0

for game_id in GAME_IDS:
    # Fetch boxscore from the NHL Web API
    try:
        r = httpx.get(f"{NHL_BASE}/gamecenter/{game_id}/boxscore", timeout=30)
        r.raise_for_status()
        raw = r.json()
    except Exception as exc:
        print(f"  SKIP {game_id}: API error: {exc}")
        failed += 1
        time.sleep(0.05)  # 50 ms rate-limiting even on failure
        continue

    # Upsert all three fact tables; commit once per game for crash safety
    try:
        with Session() as session:
            upsert_game_stats(session, game_id, raw)
            n_skaters = upsert_skater_stats(session, game_id, raw)
            n_goalies = upsert_goalie_stats(session, game_id, raw)
            session.commit()  # commit once per game — limits data loss on interruption
        processed += 1
        print(f"  OK {game_id}: skaters={n_skaters}, goalies={n_goalies}")
    except Exception as exc:
        print(f"  FAIL {game_id}: persist error: {exc}")
        failed += 1

    time.sleep(0.05)  # 50 ms rate-limiting between requests — polite to the NHL API

print()
print(f"Batch complete — processed: {processed}, failed: {failed}")

  OK 1917020001: skaters=16, goalies=2
  OK 1917020002: skaters=15, goalies=3
  OK 1917020003: skaters=14, goalies=3
  OK 1917020004: skaters=16, goalies=2
  OK 1917020005: skaters=17, goalies=2
  OK 1917020006: skaters=14, goalies=2
  OK 1917020007: skaters=16, goalies=2
  OK 1917020008: skaters=15, goalies=2
  OK 1917020009: skaters=16, goalies=2
  OK 1917020035: skaters=0, goalies=0
  OK 1917020010: skaters=16, goalies=2
  OK 1917020036: skaters=0, goalies=0
  OK 1917020011: skaters=15, goalies=3
  OK 1917020012: skaters=17, goalies=2
  OK 1917020013: skaters=17, goalies=3
  OK 1917020014: skaters=15, goalies=2
  OK 1917020015: skaters=16, goalies=3
  OK 1917020016: skaters=15, goalies=2
  OK 1917020017: skaters=16, goalies=2
  OK 1917020018: skaters=15, goalies=2
  OK 1917020019: skaters=17, goalies=2
  OK 1917020020: skaters=14, goalies=2
  OK 1917020021: skaters=15, goalies=2
  OK 1917020022: skaters=15, goalies=2
  OK 1917020023: skaters=16, goalies=2
  OK 1917020024: skaters=17

KeyboardInterrupt: 

## Section 5 — Verification

Queries all three fact tables to confirm the backfill succeeded:

1. **Row counts** for `fact_boxscore_game_stats`, `fact_skater_stats`, `fact_goalie_stats`
2. **Sample rows** from each table — spot-check game IDs, player IDs, and key fields

In [ ]:
with engine.connect() as conn:
    cnt_game   = conn.execute(text("SELECT COUNT(*) FROM fact_boxscore_game_stats")).scalar()
    cnt_skater = conn.execute(text("SELECT COUNT(*) FROM fact_skater_stats")).scalar()
    cnt_goalie = conn.execute(text("SELECT COUNT(*) FROM fact_goalie_stats")).scalar()

    sample_game = conn.execute(text(
        "SELECT game_id, game_date, away_abbrev, home_abbrev, away_score, home_score, "
        "game_state FROM fact_boxscore_game_stats ORDER BY game_date DESC LIMIT 5"
    )).fetchall()

    sample_skater = conn.execute(text(
        "SELECT game_id, player_id, side, position, goals, assists, points, toi "
        "FROM fact_skater_stats LIMIT 5"
    )).fetchall()

    sample_goalie = conn.execute(text(
        "SELECT game_id, player_id, side, starter, saves, shots_against, save_pct, decision "
        "FROM fact_goalie_stats LIMIT 5"
    )).fetchall()

print("Row counts:")
print(f"  fact_boxscore_game_stats : {cnt_game}")
print(f"  fact_skater_stats        : {cnt_skater}")
print(f"  fact_goalie_stats        : {cnt_goalie}")
print()

df_game_sample = pd.DataFrame(
    sample_game,
    columns=["game_id", "game_date", "away", "home", "away_score", "home_score", "game_state"],
)
print("Sample — fact_boxscore_game_stats (5 most recent):")
display(df_game_sample)

df_skater_sample = pd.DataFrame(
    sample_skater,
    columns=["game_id", "player_id", "side", "position", "goals", "assists", "points", "toi"],
)
print("Sample — fact_skater_stats (5 rows):")
display(df_skater_sample)

df_goalie_sample = pd.DataFrame(
    sample_goalie,
    columns=["game_id", "player_id", "side", "starter", "saves", "shots_against", "save_pct", "decision"],
)
print("Sample — fact_goalie_stats (5 rows):")
display(df_goalie_sample)